# School Funding Comparison

## Project Overview
Analyze school funding and performance metrics to compare resource allocation and outcomes across schools or districts.

## Learning Objectives
- Compare funding levels across school types and regions
- Correlate per-pupil spending with academic outcomes
- Identify funding disparities and efficiency patterns
- Analyze the role of socioeconomic context in the funding-outcome relationship

## Problem Statement
Education policymakers need to understand whether increased funding translates to better outcomes and where the biggest disparities exist to guide equitable resource allocation.

## Why This Project Matters
School funding is one of the most debated topics in education policy. Data-driven analysis separates real patterns from political narratives and helps direct resources where they matter most.

## Dataset Overview
Synthetic dataset of 400 schools across 4 regions with per-pupil spending, student demographics, test scores, graduation rates, and school type.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 400
region = np.random.choice(['Northeast', 'Southeast', 'Midwest', 'West'], n, p=[0.25, 0.25, 0.25, 0.25])
school_type = np.random.choice(['Elementary', 'Middle', 'High'], n, p=[0.40, 0.30, 0.30])
enrollment = np.clip(np.random.normal(600, 250, n).astype(int), 100, 2000)
pct_free_lunch = np.clip(np.random.beta(2, 3, n) * 100, 5, 95).round(1)

# Per-pupil spending varies by region and SES
region_bonus = {'Northeast': 3000, 'West': 2000, 'Midwest': 0, 'Southeast': -1000}
per_pupil = np.array([
    np.clip(12000 + region_bonus[r] - pct_free_lunch[i] * 30 + np.random.normal(0, 2000), 6000, 25000)
    for i, r in enumerate(region)
]).round(0)

# Outcomes: driven by spending + SES
math_score = np.clip(50 + per_pupil / 1000 * 2 - pct_free_lunch * 0.3 + np.random.normal(0, 8, n), 20, 100).round(1)
reading_score = np.clip(52 + per_pupil / 1000 * 1.8 - pct_free_lunch * 0.25 + np.random.normal(0, 7, n), 20, 100).round(1)
grad_rate = np.clip(70 + per_pupil / 1000 * 1.5 - pct_free_lunch * 0.2 + np.random.normal(0, 6, n), 40, 99).round(1)
teacher_ratio = np.clip(18 - per_pupil / 5000 + np.random.normal(0, 2, n), 8, 30).round(1)

df = pd.DataFrame({
    'SchoolID': [f'SCH{i:04d}' for i in range(n)],
    'Region': region, 'SchoolType': school_type, 'Enrollment': enrollment,
    'PctFreeLunch': pct_free_lunch, 'PerPupilSpending': per_pupil,
    'MathScore': math_score, 'ReadingScore': reading_score,
    'GradRate': grad_rate, 'StudentTeacherRatio': teacher_ratio
})
print(f'Dataset: {df.shape}')
print(f'Spending range: ${df["PerPupilSpending"].min():,.0f} — ${df["PerPupilSpending"].max():,.0f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nSchools by region: {df["Region"].value_counts().to_dict()}')
print(f'Schools by type: {df["SchoolType"].value_counts().to_dict()}')

## Funding Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['PerPupilSpending'].hist(bins=25, ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Per-Pupil Spending Distribution')
axes[0].set_xlabel('Spending ($)')
sns.boxplot(data=df, x='Region', y='PerPupilSpending', ax=axes[1])
axes[1].set_title('Spending by Region')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'funding_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Mean spending by region:')
print(df.groupby('Region')['PerPupilSpending'].mean().sort_values(ascending=False).round(0))

## Spending vs Academic Outcomes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, title in zip(axes, ['MathScore', 'ReadingScore', 'GradRate'],
                            ['Math Score', 'Reading Score', 'Graduation Rate']):
    ax.scatter(df['PerPupilSpending'], df[col], alpha=0.3, s=15)
    z = np.polyfit(df['PerPupilSpending'], df[col], 1)
    ax.plot(np.sort(df['PerPupilSpending']), np.polyval(z, np.sort(df['PerPupilSpending'])), 'r-', lw=2)
    r = df['PerPupilSpending'].corr(df[col])
    ax.set_title(f'Spending vs {title} (r={r:.2f})')
    ax.set_xlabel('Per-Pupil Spending ($)')
    ax.set_ylabel(title)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'spending_vs_outcomes.png'), dpi=100, bbox_inches='tight')
plt.show()

## Socioeconomic Context (Free Lunch %)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.scatter(df['PctFreeLunch'], df['PerPupilSpending'], alpha=0.3, s=15, color='coral')
z = np.polyfit(df['PctFreeLunch'], df['PerPupilSpending'], 1)
ax.plot(np.sort(df['PctFreeLunch']), np.polyval(z, np.sort(df['PctFreeLunch'])), 'r-', lw=2)
ax.set_title(f'Free Lunch % vs Spending (r={df["PctFreeLunch"].corr(df["PerPupilSpending"]):.2f})')
ax.set_xlabel('% Free Lunch')
ax.set_ylabel('Per-Pupil Spending ($)')

ax = axes[1]
ax.scatter(df['PctFreeLunch'], df['MathScore'], alpha=0.3, s=15, color='teal')
z = np.polyfit(df['PctFreeLunch'], df['MathScore'], 1)
ax.plot(np.sort(df['PctFreeLunch']), np.polyval(z, np.sort(df['PctFreeLunch'])), 'r-', lw=2)
ax.set_title(f'Free Lunch % vs Math Score (r={df["PctFreeLunch"].corr(df["MathScore"]):.2f})')
ax.set_xlabel('% Free Lunch')
ax.set_ylabel('Math Score')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'ses_context.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Outcome Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['MathScore', 'ReadingScore', 'GradRate']):
    sns.boxplot(data=df, x='Region', y=col, ax=ax)
    ax.set_title(f'{col} by Region')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional_outcomes.png'), dpi=100, bbox_inches='tight')
plt.show()

## Efficiency Analysis: Spending per Score Point

In [ ]:
df['CostPerMathPoint'] = (df['PerPupilSpending'] / df['MathScore']).round(1)
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='Region', y='CostPerMathPoint', ax=ax)
ax.set_title('Cost per Math Score Point by Region')
ax.set_ylabel('$ per Math Point')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'efficiency.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Median cost per math point by region:')
print(df.groupby('Region')['CostPerMathPoint'].median().sort_values(ascending=False).round(1))

## Interpretation and Business Insight
- **Higher spending correlates with better outcomes**, but the relationship is moderate — money alone doesn't guarantee results
- **Socioeconomic status** (free lunch %) is a stronger predictor of outcomes than spending alone
- **Regional disparities** exist in both funding and outcomes — Southeast schools receive less and score lower
- **Efficiency varies** — some regions achieve similar outcomes at lower cost, suggesting better resource utilization
- **Student-teacher ratio** decreases with higher spending, providing a plausible mechanism for the spending-outcome link
- Policy should target both funding equity AND effective resource deployment

## Limitations
- Synthetic data with simplified causal model
- No school-level leadership or curriculum quality data
- No longitudinal tracking
- No within-district variation
- Socioeconomic proxy (free lunch %) is a single rough measure

## How to Improve This Project
- Add longitudinal funding and outcome tracking
- Include teacher quality metrics (certification, experience)
- Build regression models controlling for SES to isolate funding effects
- Add within-district school comparisons
- Include state policy variables (testing standards, accountability)

## Production Considerations
- District-level funding equity dashboards
- Predictive models linking resource allocation to expected outcomes
- Automated disparity flagging for state education agencies
- Benchmarking tools for peer school comparison

## Common Mistakes
- Concluding that spending doesn't matter because correlation is imperfect
- Ignoring SES as a confounding variable
- Comparing schools without adjusting for demographic context
- Using averages that mask within-region or within-district variation

## Mini Challenge / Exercises
1. Which region has the smallest gap between highest and lowest funded schools?
2. Among schools with > 60% free lunch, what is the correlation between spending and math score?
3. Find the top 10 'most efficient' schools (highest math score per dollar spent). What do they have in common?
4. If per-pupil spending increased by $2,000 for all schools, estimate the expected math score improvement using the linear relationship.

## Final Summary / Key Takeaways
- School funding matters, but socioeconomic context matters more
- Regional funding disparities create structural inequities in outcomes
- Efficiency analysis reveals that smart resource allocation can compensate for lower absolute spending
- Equity-focused policy should address both funding levels and deployment effectiveness
- Simple spending comparisons without SES adjustment are misleading